> **Known incorrect — do not use as a reference for SMA.**
>
> The filter below is named for a simple moving average but implements an
> exponential one, and it has none of the shipped `SmaFilter`'s warmup or
> gap-reset semantics.
>
> The correct implementation is [`../bartons/src/kernels/sma.rs`](../bartons/src/kernels/sma.rs).
> This notebook is kept as prototyping history; it needs a rewrite before any
> conclusion is drawn from it.

In [3]:
let ov : Vec<Option<f64>> = vec![Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)];
ov

[Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)]

In [4]:
ov.iter().map(|opt| opt.map(|x| x + 1.0)).collect::<Vec<_>>()

[Some(2.0), Some(3.0), None, Some(5.0), Some(6.0)]

In [5]:
let fv : Vec<f64> = vec![1.0, 2.0, f64::NAN, 4.0, 5.0];
fv

[1.0, 2.0, NaN, 4.0, 5.0]

In [6]:
fv.iter().map (
    |x| if f64::is_nan(*x) { None } else { Some(*x) }
).collect::<Vec<_>>()


[Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)]

In [27]:
use std::fmt;

pub struct SimpleMovingAverage {
    period: i64,
    alpha: f64,
    current: f64,
    count: i64,
}

impl SimpleMovingAverage {
    
    pub fn new(period: i64) -> Result<Self, String> {
        if period <= 0 {
            return Err("Period must be positive".to_string());
        }
        
        Ok(Self {
            period,
            alpha: 2.0 / (period + 1) as f64,
            current: 0.0,
            count: 0,
        })
    }

    fn next(&mut self, input: f64) -> f64 {
        if self.count == 0 {
            self.current = input;
        } else {
            self.current = self.alpha * input + (1.0 - self.alpha) * self.current;
        }
        self.count += 1;
        self.current
    }

    fn next_option(&mut self, input: Option<f64>) -> Option<f64> {
        if input.is_none() {
            return None;
        }

        let result = self.next(input.unwrap());
        Some(result)
    }
}

In [28]:
let fv : Vec<f64> = vec![1.0, 2.0, 3.0, 4.0, 5.0];

let mut sma = SimpleMovingAverage::new(3).unwrap();

fv.iter().map (
    |x| sma.next(*x)
).collect::<Vec<_>>()


[1.0, 1.5, 2.25, 3.125, 4.0625]

In [29]:
let ov : Vec<Option<f64>> = vec![Some(1.0), Some(2.0), None, Some(4.0), Some(5.0)];

let mut sma = SimpleMovingAverage::new(3).unwrap();

ov.iter().map(
    |x| sma.next_option(*x)
).collect::<Vec<_>>()


[Some(1.0), Some(1.5), None, Some(2.75), Some(3.875)]